# Processing time of the multiplexed-input methods

Wall-clock time per image, on CPU and GPU, for four ways of feeding a multiplexed stack to a model:
MIP (max-projection), LLC (two 1x1 conv layers in front of the backbone), MAM (widened first layer(s)) and
ChannelNet-tuned (AdaptorNet in front of the segmenter). Four models, four datasets, every image of each dataset,
datasets weighted equally.

Each image's time is split into **Loading** (reading the file; MIP reads the smaller cached `MaxProj.png` for MACSima),
**Preprocessing** (MIP: `max_project`; LLC / MAM: only the added layers; ChannelNet-tuned: the AdaptorNet) and
**Forward** (the segmentation backbone's forward pass; decoding to labels is not timed).

- ChannelNet-tuned is the only method whose Preprocessing and Forward are separate modules, so they add up. For LLC and
  MAM the added layers sit inside the backbone, so Preprocessing is measured by calling them a second time and
  Preprocessing + Forward overlaps instead of summing.
- The weights are the pretrained Stage-2 backbones with freshly initialised adapters, not trained checkpoints:
  time depends on the compute graph, not on the weight values.
- The channel-selection step (gathering and zero-padding the raw stack) is not timed.

In [ ]:
import importlib
import sys
import time
from pathlib import Path

BENCH_DIR = next(p for p in [Path.cwd()] + list(Path.cwd().parents) if p.name == "bench")
sys.path.insert(0, str(BENCH_DIR))

import numpy as np
import pandas as pd
import skimage.io
import skimage.transform
import torch

import figures.render_utils as ru
from data.Stacked_Multiplexed import max_project
from import_util import data_path
from inferencing.utils import preproc_img, preproc_stack
from training.common import channels as ch

EXPORT_DIR = BENCH_DIR / "figures" / "export"
TRAINING = BENCH_DIR / "training"

DATASETS = ["Vectra", "CODEX", "MACSima", "Zeiss"]
MODELS = ["Dist U-Net", "Stardist", "Cellpose", "InstanSeg"]
METHODS = ["MIP", "LLC", "MAM", "ChannelNetTuned"]
GPU = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"
DEVICES = ["cpu", GPU] if torch.cuda.is_available() else ["cpu"]
IMG_SIZE = (256, 256)
N_WARMUP = 3     # untimed GPU passes per (model, dataset) before timing starts
N_IMAGES = None  # set to an int to time only the first N images of each dataset

PRETRAINED = {  # Stage-2 pretrained backbone of each model
    "Dist U-Net": TRAINING / "UNet/Stage_2/DIST_STAGE2.ckpt",
    "Stardist": TRAINING / "CellViT_Star/Stage_2/checkpoints/Stardist_STAGE2_BEST.pt",
    "Cellpose": TRAINING / "Cellpose/Stage_2/CPose2ch_STAGE2_full_BEST.pt",
    "InstanSeg": TRAINING / "InstanSeg/Stage_2/checkpoints/InstanSeg_STAGE2_BEST.pt",
}
MAM_MODULE = {  # run_multiplexed_input.build_model(n_ch) builds the widened-first-layer model
    model: f"training.{d}.Experiments.Vectra.run_multiplexed_input"
    for model, d in [("Dist U-Net", "UNet"), ("Stardist", "CellViT_Star"), ("Cellpose", "Cellpose"), ("InstanSeg", "InstanSeg")]
}
MIP_N_CH = {"Dist U-Net": 3, "Stardist": 3, "Cellpose": 2, "InstanSeg": 3}  # channels each model takes as MIP input
DATASET_NAMES = {
    "Vectra": list(ch.VECTRA_DAPI), "CODEX": list(ch.CODEX_SELECTED),
    "MACSima": list(ch.MACSIMA_SELECTED_MARKERS), "Zeiss": list(ch.ZEISS_SELECTED),
}

## Build the four variants of each model

In [ ]:
def place(net, model, device, tuned=False):
    """Move `net` to `device`. Cellpose's wrapper keeps its network in `.net` and fixes `mkldnn` at construction,
    which breaks on CUDA. Some builders leave sub-modules on other devices, so go through the CPU first."""
    module = net if tuned or model != "Cellpose" else net.net
    if model == "Cellpose" and device != "cpu":
        (net.segmenter if tuned else net.net).mkldnn = False
    module.to("cpu")
    module.to(device)
    return net


def build_plain(model, device):
    pretrained = str(PRETRAINED[model])
    if model == "Dist U-Net":
        from modelling.UNet.DistNet import DIST_Net_Lightning
        net = DIST_Net_Lightning.load_from_checkpoint(
            arch="unet", encoder_name="resnet50", in_channels=3, out_classes=2,
            checkpoint_path=pretrained, map_location="cpu")
    elif model == "Stardist":
        from modelling.Cellvit.StarDistRN50_Rnd import StarDistRN50_Rnd
        net = StarDistRN50_Rnd(n_seg_cls=1, device=device)
        net.load_model(pretrained)
    elif model == "Cellpose":
        from modelling.Cellpose.myCellpose import myCellpose
        net = myCellpose(nchan=2, gpu=(device != "cpu"), pretrained_model=pretrained)
    elif model == "InstanSeg":
        from modelling.InstanSeg.InstanSegNet import build_fresh
        net = build_fresh(device=device)
        net.load_model(pretrained, device=device)
    return place(net, model, device)


def build_llc(model, n_ch, device):
    spec = ru.RENDER_1X1_SPECS[model]
    return place(spec["build"](spec["pretrained"], n_ch), model, device)


def build_mam(model, n_ch, device):
    net = importlib.import_module(MAM_MODULE[model]).build_model(n_ch)
    return place(net, model, device)


def build_tuned(model, device):
    pretrained = str(PRETRAINED[model])
    if model == "Dist U-Net":
        from modelling.UNet.AdaptorDistNet import build
        net = build(pretrained)
    elif model == "Stardist":
        from modelling.Cellvit.AdaptorStarDistNet import build
        net = build(pretrained, device=device)
    elif model == "Cellpose":
        from modelling.Cellpose.myCellpose import myCellpose
        from modelling.Cellpose.AdaptorCellposeNet import AdaptorNetCellpose
        net = AdaptorNetCellpose(myCellpose(nchan=2, gpu=(device != "cpu"), pretrained_model=pretrained))
    elif model == "InstanSeg":
        from modelling.InstanSeg.AdaptorInstanSegNet import build
        net = build(pretrained, device=device)
    return place(net, model, device, tuned=True)


def forward_module(model, net, tuned=False):
    """The backbone whose forward pass is timed (for ChannelNet-tuned: the segmenter, not the AdaptorNet)."""
    base = net.segmenter if tuned else net
    if model == "Dist U-Net":
        return base.model
    if model == "Stardist":
        return base
    if model == "Cellpose":
        return base if tuned else base.net  # the tuned segmenter already is the raw network
    if model == "InstanSeg":
        return base.backbone


def mam_layers(model, net):
    """The layers that were replaced to accept the extra channels."""
    if model == "Dist U-Net":
        return [net.model.encoder.conv1]
    if model == "Stardist":
        return [net.encoder.conv1]
    if model == "Cellpose":
        block = net.net.downsample.down.res_down_0
        return [block.conv.conv_0, block.proj]
    if model == "InstanSeg":
        block = net.backbone.encoder[0]
        return [block.conv0, block.conv1]


def n_params(*modules):
    return sum(p.numel() for m in modules for p in m.parameters())

## Prepare the inputs

In [ ]:
def reshape_to_hwc(stack):
    if stack.ndim > 3:
        n, h, w, c = stack.shape
        return np.moveaxis(stack, 0, -1).reshape(h, w, n * c)
    if stack.shape[0] < stack.shape[1] and stack.shape[0] < stack.shape[2]:
        return np.transpose(stack, (1, 2, 0))
    return stack


def resize(img):
    return skimage.transform.resize(img, IMG_SIZE, preserve_range=True, anti_aliasing=True).astype(np.float32)


def normalize(img):
    """Per-channel min-max to [0, 1]."""
    lo, hi = img.min(axis=(0, 1)), img.max(axis=(0, 1))
    return (img - lo) / np.where(hi - lo == 0, 1, hi - lo)


def model_input(model, img, device):
    """Raw HWC image -> (1, C, H, W) tensor, preprocessed the way the model's inference code does it."""
    if model in ("Dist U-Net", "InstanSeg"):
        h, w, _ = img.shape
        img = np.pad(img, ((0, -h % 32), (0, -w % 32), (0, 0)))
        x = torch.from_numpy(normalize(img).astype(np.float32)).permute(2, 0, 1).unsqueeze(0)
    elif model == "Stardist":
        x = torch.from_numpy(preproc_img(img)).unsqueeze(0).permute(0, 3, 1, 2)
    elif model == "Cellpose":
        x = torch.from_numpy(preproc_stack(img)).permute(2, 0, 1).unsqueeze(0).float()
    return x.to(device)


def ms_since(t0):
    return (time.perf_counter() - t0) * 1000

## Parameters of the added layers

Parameter count of what each method adds in front of the backbone: MIP none; LLC the two 1x1 layers; MAM the extra
parameters of the widened first layer(s) compared with the native 3-channel (Cellpose: 2-channel) one;
ChannelNet-tuned the AdaptorNet. LLC and MAM depend on the number of input channels (8 on the selected arm).

In [ ]:
def value_range(values):
    lo, hi = min(values), max(values)
    return str(lo) if lo == hi else f"{lo}-{hi}"


param_counts = {}
for model in MODELS:
    param_counts[(model, "MIP")] = "0"

    llc = [n_params(*ru._1x1_layers(build_llc(model, n_ch, "cpu"), model)) for n_ch in {ch.n_input_channels(d, "selected", "1x1") for d in DATASETS}]
    param_counts[(model, "LLC")] = value_range(llc)

    native = n_params(*mam_layers(model, build_plain(model, "cpu")))
    mam = [n_params(*mam_layers(model, build_mam(model, n_ch, "cpu"))) - native for n_ch in {ch.n_input_channels(d, "selected", "1x1") for d in DATASETS}]
    param_counts[(model, "MAM")] = value_range(mam)

    param_counts[(model, "ChannelNetTuned")] = str(n_params(build_tuned(model, "cpu").adaptor_net))
param_counts

## Time every stage

In [ ]:
def time_stages(model, nets, x_mip, x_llc, x_adapt, device):
    """Time the added layers and the backbone forward pass of every method once; returns two {method: ms} dicts."""
    def timed(fn, x):
        with torch.no_grad():
            if device != "cpu":
                torch.cuda.synchronize(device)
            t0 = time.perf_counter()
            out = fn(x)
            if device != "cpu":
                torch.cuda.synchronize(device)
        return ms_since(t0), out

    layer1, layer2 = ru._1x1_layers(nets["LLC"], model)
    mam = mam_layers(model, nets["MAM"])

    prep, forward = {}, {}
    prep["ChannelNetTuned"], adapted = timed(nets["ChannelNetTuned"].adaptor_net, x_adapt)
    prep["LLC"], _ = timed(lambda x: layer2(layer1(x)), x_llc)
    prep["MAM"], _ = timed(lambda x: [layer(x) for layer in mam], x_llc)
    forward["MIP"], _ = timed(forward_module(model, nets["MIP"]), x_mip)
    forward["LLC"], _ = timed(forward_module(model, nets["LLC"]), x_llc)
    forward["MAM"], _ = timed(forward_module(model, nets["MAM"]), x_llc)
    forward["ChannelNetTuned"], _ = timed(forward_module(model, nets["ChannelNetTuned"], tuned=True), adapted)
    return prep, forward


rows = []
for model in MODELS:
    for dataset in DATASETS:
        n_ch = ch.n_input_channels(dataset, "selected", "1x1")
        img_paths, _, stack_funcs, _ = ch.build_paths(
            dataset, DATASET_NAMES[dataset], str(data_path(f"Multiplex/{dataset}/")), "selected", "1x1")
        images = list(zip(img_paths, stack_funcs))[:N_IMAGES]

        for device in DEVICES:
            nets = {"MIP": build_plain(model, device), "LLC": build_llc(model, n_ch, device),
                    "MAM": build_mam(model, n_ch, device), "ChannelNetTuned": build_tuned(model, device)}

            warmed_up = False
            for raw_path, stack_func in images:
                # Loading. MIP reads the smaller cached MaxProj.png for MACSima; everything else reads the raw stack.
                t0 = time.perf_counter()
                raw_stack = skimage.io.imread(raw_path)
                load = {m: ms_since(t0) for m in METHODS}
                if dataset == "MACSima":
                    t0 = time.perf_counter()
                    mip_image = skimage.io.imread(str(Path(raw_path).parent / ch.image_filename("MACSima", "selected", "max_proj")))
                    load["MIP"] = ms_since(t0)
                    prep_mip = 0.0
                else:
                    t0 = time.perf_counter()
                    mip_image = max_project(reshape_to_hwc(raw_stack))
                    prep_mip = ms_since(t0)

                # Inputs (not timed): the channel gather, resize and per-model preprocessing
                gathered = resize(stack_func(reshape_to_hwc(raw_stack)))
                x_adapt = torch.from_numpy(normalize(gathered).astype(np.float32)).permute(2, 0, 1).unsqueeze(0).to(device)
                x_llc = model_input(model, gathered, device)
                x_mip = model_input(model, resize(mip_image)[:, :, :MIP_N_CH[model]], device)

                if device != "cpu" and not warmed_up:
                    for _ in range(N_WARMUP):
                        time_stages(model, nets, x_mip, x_llc, x_adapt, device)
                    warmed_up = True

                prep, forward = time_stages(model, nets, x_mip, x_llc, x_adapt, device)
                prep["MIP"] = prep_mip
                for method in METHODS:
                    rows.append(dict(Model=model, Dataset=dataset, Device=device, Method=method,
                                     Loading=load[method], Preprocessing=prep[method], Forward=forward[method]))

            print(f"done: {model} / {dataset} / {device} ({len(images)} images)")
            del nets
            if device != "cpu":
                torch.cuda.empty_cache()

raw = pd.DataFrame(rows)
raw.to_csv(EXPORT_DIR / "method_timing_raw.csv", index=False)
raw.head()

## Summary table

Mean over the images of each dataset, then over the four datasets.

In [ ]:
per_dataset = raw.groupby(["Model", "Method", "Device", "Dataset"])[["Loading", "Preprocessing", "Forward"]].mean()
by_device = per_dataset.groupby(["Model", "Method", "Device"]).mean()
has_gpu = GPU in DEVICES

rows = []
for model in MODELS:
    for method in METHODS:
        cpu = by_device.loc[(model, method, "cpu")]
        gpu = by_device.loc[(model, method, GPU)] if has_gpu else None
        rows.append({
            "Algorithm": model,
            "Preprocessing Method": method,
            "N Preprocessing Parameters": param_counts[(model, method)],
            "Loading (ms)": round(cpu["Loading"], 3),
            "CPU Preprocessing Method (ms)": round(cpu["Preprocessing"], 3),
            "GPU Preprocessing Method (ms)": "N/A" if method == "MIP" or not has_gpu else round(gpu["Preprocessing"], 3),
            "CPU Model inference (ms)": round(cpu["Forward"], 3),
            "GPU Model inference (ms)": round(gpu["Forward"], 3) if has_gpu else "N/A",
        })
summary = pd.DataFrame(rows)
summary.to_csv(EXPORT_DIR / "method_timing_summary.csv", index=False)
summary